# 02 · Oferta escolar — INEP 2025

**Objetivo.** Auditar as escolas de Lavras, caracterizar a rede pública por etapa e produzir **cenários explícitos de capacidade**, sem tratar matrículas como capacidade física.

- Matrículas \(M_{je}\): demanda **atendida observada**, não capacidade máxima.
- Turmas \(T_{je}\): indicador operacional; o total de turmas pode não representar salas simultâneas disponíveis.
- Lotação hipotética \(s_e\): parâmetro de cenário, não norma nem capacidade licenciada.
- Capacidade proxy: \(q_{je}(s_e)=T_{je}s_e\). Escolas com matrícula positiva e turmas ausentes são bloqueadas para revisão.

O modelo não afirma disponibilidade de terrenos ou capacidade de construção de novas escolas.

## 1. Carregar matrículas e turmas para Lavras

Mantemos escolas públicas ativas (federal, estadual e municipal), com matrícula positiva na etapa. O registro identificado como CENAV já havia sido excluído de modo documentado na versão anterior; essa exclusão permanece por hipótese de comparabilidade e deve ser reavaliada se o escopo administrativo mudar.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

raiz = Path.cwd().resolve()
if not (raiz / "pyproject.toml").exists():
    raiz = next((p for p in raiz.parents if (p / "pyproject.toml").exists()), raiz)
assert (raiz / "pyproject.toml").exists(), "Execute dentro do repositório PM."
saida = raiz / "outputs" / "localizacao_escolar" / "lavras"
saida.mkdir(parents=True, exist_ok=True)
CODIGO_MUNICIPIO = "3138203"
ETAPAS = {"fundamental_iniciais": ("V01032", "QT_MAT_FUND_AI", "QT_TUR_FUND_AI"),
          "fundamental_finais": ("V01033", "QT_MAT_FUND_AF", "QT_TUR_FUND_AF"),
          "medio": ("V01034", "QT_MAT_MED_PROP", "QT_TUR_MED_PROP")}
fontes = list((raiz / "data").rglob("Tabela_Escola_2025_V2.csv"))
assert len(fontes) == 1, "Localize exatamente uma cópia dos microdados INEP 2025 V2"
pasta_inep = fontes[0].parent
arquivo_catalogo = raiz / "data" / "Análise - Tabela da lista das escolas - Detalhado.csv"
assert arquivo_catalogo.exists(), "Falta catálogo de escolas INEP para as coordenadas"

In [ ]:
def ler_municipio(arquivo, campos):
    fatias = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype="string",
                         usecols=["CO_MUNICIPIO", *campos], chunksize=20_000)
    return pd.concat([b.loc[b.CO_MUNICIPIO.eq(CODIGO_MUNICIPIO)] for b in fatias],
                     ignore_index=True).drop(columns="CO_MUNICIPIO")

escolas = ler_municipio(fontes[0],
                       ["CO_ENTIDADE", "NO_ENTIDADE", "TP_DEPENDENCIA", "TP_SITUACAO_FUNCIONAMENTO"])
for nome, idx in [("Tabela_Matricula_2025_V2.csv", 1),
                  ("Tabela_Turma_2025_V2.csv", 2)]:
    arq = pasta_inep / nome
    assert arq.exists(), f"Microdado não encontrado: {arq}"
    cols = ["CO_ENTIDADE"] + [val[idx] for val in ETAPAS.values()]
    tab = ler_municipio(arq, cols)
    escolas = escolas.merge(tab, on="CO_ENTIDADE", how="left", validate="one_to_one")
assert escolas.CO_ENTIDADE.is_unique
for campo in [val[idx] for val in ETAPAS.values() for idx in (1, 2)]:
    escolas[campo] = pd.to_numeric(escolas[campo], errors="raise")
escolas["publica_ativa"] = escolas.TP_DEPENDENCIA.isin(["1", "2", "3"]) & escolas.TP_SITUACAO_FUNCIONAMENTO.eq("1")
escolas["exclusao_auditada"] = escolas.CO_ENTIDADE.map({"31347930": "CENAV: atendimento especializado; reavaliar regra"})
for etapa, (_, mat, tur) in ETAPAS.items():
    escolas["elegivel_" + etapa] = escolas.publica_ativa & escolas[mat].gt(0) & escolas.exclusao_auditada.isna()
    assert not (escolas["elegivel_" + etapa] & (escolas[tur].isna() | escolas[tur].le(0))).any(), f"Revisar turmas nulas/zero: {etapa}"
display(escolas[["CO_ENTIDADE", "NO_ENTIDADE", "publica_ativa", *["elegivel_" + e for e in ETAPAS]]].head())
print("Escolas no cadastro municipal:", len(escolas))

## 2. Coordenadas — catálogo INEP e complementos aprovados

Sem coordenada geográfica confiável, a escola não pode entrar inadvertidamente na matriz espacial. Registros pendentes aparecem na auditoria, em vez de serem excluídos silenciosamente. Correções de Lavras existentes localmente podem ser usadas, desde que guardem fonte e precisão.

In [ ]:
catalogo = pd.read_csv(arquivo_catalogo, dtype="string",
                       usecols=["Código INEP", "Latitude", "Longitude", "Endereço"])
catalogo = catalogo.rename(columns={"Código INEP": "CO_ENTIDADE"})
assert catalogo.CO_ENTIDADE.is_unique
for dst, src in [("latitude", "Latitude"), ("longitude", "Longitude")]:
    catalogo[dst] = pd.to_numeric(catalogo[src].str.replace(",", ".", regex=False), errors="coerce")
escolas = escolas.merge(catalogo[["CO_ENTIDADE", "latitude", "longitude", "Endereço"]],
                        on="CO_ENTIDADE", how="left", validate="one_to_one")
escolas["fonte_coordenada"] = "catalogo_inep"
escolas["precisao_coordenada"] = "nao_documentada"
caminho_aprovadas = raiz / "outputs" / "3138203" / "escolas" / "geocodificacao_aprovada.csv"
if caminho_aprovadas.exists():
    aprovadas = pd.read_csv(caminho_aprovadas, dtype={"CO_ENTIDADE": "string"})
    for row in aprovadas.itertuples(index=False):
        mask = escolas.CO_ENTIDADE.eq(row.CO_ENTIDADE) & (escolas.latitude.isna() | escolas.longitude.isna())
        escolas.loc[mask, "latitude"] = float(row.latitude)
        escolas.loc[mask, "longitude"] = float(row.longitude)
        escolas.loc[mask, "fonte_coordenada"] = getattr(row, "fonte", "complemento_auditado")
        escolas.loc[mask, "precisao_coordenada"] = getattr(row, "precisao", "nao_documentada")
escolas["coord_valida"] = (escolas.latitude.between(-90, 90) & escolas.longitude.between(-180, 180)).fillna(False)
auditoria = []
for etapa in ETAPAS:
    conjunto = escolas.loc[escolas["elegivel_" + etapa]]
    auditoria.append({"etapa": etapa, "escolas_elegiveis": len(conjunto),
                      "matriculas": int(conjunto[ETAPAS[etapa][1]].sum()),
                      "turmas": int(conjunto[ETAPAS[etapa][2]].sum()),
                      "sem_coordenadas": int((~conjunto.coord_valida).sum())})
auditoria = pd.DataFrame(auditoria)
display(auditoria)

## 3. Auditoria de infraestrutura: salas e turnos

A capacidade real não pode ser inferida apenas pelas matrículas. O Censo Escolar informa indicadores de infraestrutura, inclusive salas de aula utilizadas dentro e fora da escola, e de turmas. Estes campos **não são vagas oficiais**. Um mesmo espaço físico pode ser utilizado em horários diferentes; além disso, salas fora do prédio podem ser compartilhadas. Não multiplicar automaticamente salas por turnos.

Este diagnóstico mantém os campos disponíveis no cadastro 2025 V2. A disponibilidade efetiva dos horários individuais depende da base de turmas detalhada, que deve ser incorporada separadamente, e não pode ser inferida da simples contagem agregada de turmas.

In [ ]:
# Recuperar, se presentes no arquivo 2025, indicadores de infraestrutura física.
# Não supor que o mesmo conjunto de colunas existe em todas as versões do Censo.
header = pd.read_csv(fontes[0], sep=";", encoding="latin1", nrows=0).columns
candidatas_salas = ["QT_SALAS_EXISTENTES", "QT_SALAS_UTILIZADAS",
                    "QT_SALAS_UTILIZADAS_DENTRO", "QT_SALAS_UTILIZADAS_FORA",
                    "QT_SALAS_UTILIZA_CLIMATIZADAS", "QT_SALAS_UTILIZADAS_ACESSIVEIS"]
colunas_salas = [campo for campo in candidatas_salas if campo in header]
print("Campos de salas disponíveis:", colunas_salas)
if colunas_salas:
    infra = ler_municipio(fontes[0], ["CO_ENTIDADE", *colunas_salas])
    assert infra.CO_ENTIDADE.is_unique
    for campo in colunas_salas:
        infra[campo] = pd.to_numeric(infra[campo], errors="coerce")
    escolas = escolas.merge(infra, on="CO_ENTIDADE", how="left", validate="one_to_one")
    escolas["sem_info_salas"] = escolas[colunas_salas].isna().all(axis=1)
else:
    escolas["sem_info_salas"] = True

for etapa, (_, campo_mat, campo_turmas) in ETAPAS.items():
    rede = escolas.loc[escolas["elegivel_" + etapa]].copy()
    print("\\n", etapa, "| unidades:", len(rede),
          "| sem informação de salas:", int(rede.sem_info_salas.sum()))
    if "QT_SALAS_UTILIZADAS" in rede:
        rede["turmas_etapa_por_sala_informada"] = (
            rede[campo_turmas] / rede.QT_SALAS_UTILIZADAS.replace(0, np.nan)
        )
        display(rede[["NO_ENTIDADE", campo_turmas, "QT_SALAS_UTILIZADAS",
                      "turmas_etapa_por_sala_informada"]].head(10))
print("Alerta: turmas por sala NÃO são taxa de ocupação física, pois há turnos e etapas compartilhadas.")

### Validação entre matrículas e capacidade proxy

Antes de usar \(Q_{je}=T_{je}L_e\), comparar com a matrícula observada \(M_{je}\). Se \(M_{je}>Q_{je}\), o cenário de lotação é incompatível com a operação registrada e **não deve ser tratado como capacidade validada**. Não corrigir silenciosamente \(Q\) para igualar \(M\); isso confundiria indicador observado com limite operacional.

**Próximo refinamento recomendado:** calcular a ocupação simultânea a partir de horários de turmas detalhados e compará-la com a disponibilidade de salas da unidade. Esse cruzamento evita duplicar salas em etapas e turnos.

## 3. Cenários de capacidade

Os três parâmetros abaixo são **hipóteses operacionais de exercício**. Não correspondem à legislação escolar, à lotação aprovada nem ao limite físico das instalações. Um planejamento real teria de utilizar salas por turno, modalidades, acessibilidade, pessoal e informações de infraestrutura. Não substituímos dados ausentes por matrículas.

In [ ]:
LOTACAO_HIPOTETICA = {"conservadora": 25, "central": 30, "expansiva": 35}
linhas = []
for etapa, (_, mat, tur) in ETAPAS.items():
    elegiveis = escolas.loc[escolas["elegivel_" + etapa]].copy()
    for cenario, tamanho_turma in LOTACAO_HIPOTETICA.items():
        capacidade = elegiveis[tur] * tamanho_turma
        linhas.append({"etapa": etapa, "cenario": cenario, "escolas": len(elegiveis),
                       "matriculas_observadas": int(elegiveis[mat].sum()),
                       "capacidade_proxy": int(capacidade.sum()),
                       "turma_hipotetica": tamanho_turma,
                       "capacidade_inferior_as_matriculas": int((capacidade < elegiveis[mat]).sum())})
resumo_capacidade = pd.DataFrame(linhas)
display(resumo_capacidade)
inconsistentes = resumo_capacidade.loc[resumo_capacidade.capacidade_inferior_as_matriculas.gt(0)]
if len(inconsistentes):
    print("CENÁRIOS NÃO VALIDADOS: a capacidade por turmas é menor que as matrículas observadas")
    display(inconsistentes)
# Nenhum caso é automaticamente corrigido para a matrícula: exige avaliação do parâmetro L.
escolas.to_csv(saida / "escolas_auditadas.csv", index=False)
resumo_capacidade.to_csv(saida / "auditoria_capacidade.csv", index=False)
(saida / "metadados_inep.json").write_text(json.dumps({
    "ano_inep": 2025, "lotacao_por_turma": LOTACAO_HIPOTETICA,
    "metodo": "numero de turmas registrado multiplicado por lotacao hipotetica; nao e capacidade certificada",
    "exclusoes": {"31347930": "atendimento especializado auditado, excluido do escopo comum"},
    "coordenadas": "catalogo INEP; complemento local auditado quando presente",
    "matriculas": "uso observado, nao usadas para estimar a demanda dos setores"
}, ensure_ascii=False, indent=2))
print("Exportado:", saida / "escolas_auditadas.csv")